# VSCode Jupyter 一键自检 Notebook

**使用方式**：
1. 打开本文件后，右上角点击 `Select Kernel`，选择装了 `ipykernel` 的 Python 环境
2. 每个单元格左侧点 ▶️ 或按 `Shift+Enter` 逐格运行
3. 想一键全跑：顶部工具栏 `Run All`

## 1. 环境自检

In [1]:
import load_dotenv
from typing_extensions import override
from urllib3.contrib.emscripten import response

print("Python 版本 :hello")
print('Hello')

ModuleNotFoundError: No module named 'js'

In [ ]:
from dotenv import load_dotenv
from deepagents import create_deep_agent
from rich import print

load_dotenv(override=True)

# def get_weather(city: str) -> str:
#     """Get weather for a given city."""
#     return f"It's always sunny in {city}!"
#
#
# agent = create_deep_agent(
#     model="openai:gpt-5.5",
#     tools=[get_weather],
#     system_prompt="You are a helpful assistant",
# )
#
# # Run the agent
# response = agent.invoke(
#     {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
# )

# print(response)

# Step 3: Create a search tool

In [ ]:
import os
from typing import Literal

from tavily import TavilyClient
from deepagents import create_deep_agent

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

load_dotenv(override=True)

def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run a web search"""
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )

# Step 4: Create a deep agent

In [ ]:
# System prompt to steer the agent to be an expert researcher
research_instructions = """You are an expert researcher. Your job is to conduct thorough research and then write a polished report.

You have access to an internet search tool as your primary means of gathering information.

## `internet_search`

Use this to run an internet search for a given query. You can specify the max number of results to return, the topic, and whether raw content should be included.
"""

agent = create_deep_agent(
    model="openai:gpt-5.5",
    tools=[internet_search],
    system_prompt=research_instructions,
)

# LangSmith

In [ ]:
from rich import print

result = agent.invoke({"messages": [{"role": "user", "content": "What is langgraph?"}]})

print(result)

# Print the agent's response
print(result["messages"][-1].content)

# Custom middleware

In [ ]:
from langchain.agents.middleware import wrap_tool_call
from langchain.tools import tool
from deepagents import create_deep_agent
from rich import print


@tool
def get_weather(city: str) -> str:
    """Get the weather in a city."""
    return f"The weather in {city} is sunny."


call_count = [0]  # Use list to allow modification in nested function


@wrap_tool_call
def log_tool_calls(request, handler):
    """Intercept and log every tool call - demonstrates cross-cutting concern."""
    call_count[0] += 1
    tool_name = request.name if hasattr(request, "name") else str(request)

    print(f"[Middleware] Tool call #{call_count[0]}: {tool_name}")
    print(f"[Middleware] Arguments: {request.args if hasattr(request, 'args') else 'N/A'}")

    # Execute the tool call
    result = handler(request)

    # Log the result
    print(f"[Middleware] Tool call #{call_count[0]} completed")

    return result


agent = create_deep_agent(
    model="openai:gpt-5.5",
    tools=[get_weather],
    middleware=[log_tool_calls],
)

result = agent.invoke({"messages": [{"role": "user", "content": "北京天气怎么样,并打印日志"}]})

print(result)

# Override a default middleware instance

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import StateBackend
from deepagents.middleware import SummarizationMiddleware
from rich import print

backend = StateBackend()
model = "openai:gpt-5.5"

custom_summarization = SummarizationMiddleware(
    model=model,
    backend=backend,
    summary_prompt="Your custom summary prompt here.",
)

agent = create_deep_agent(
    model=model,
    middleware=[custom_summarization],  # replaces the default SummarizationMiddleware
)

from rich import print
result = agent.invoke({"messages": [{"role": "user", "content": "Hello"}]})
print(result)

# Interpreters

In [ ]:
from deepagents import create_deep_agent
from langchain_quickjs import CodeInterpreterMiddleware

agent = create_deep_agent(
    model="openai:gpt-5.5",
    middleware=[CodeInterpreterMiddleware()],
)


from rich import print
result = agent.invoke({"messages": [{"role": "user", "content": "Hello"}]})
print(result)

# Subagents

In [ ]:
import os
from typing import Literal

from deepagents import create_deep_agent
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run a web search"""
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )


research_subagent = {
    "name": "research-agent",
    "description": "Used to research more in depth questions",
    "system_prompt": "You are a great researcher",
    "tools": [internet_search],
    "model": "openai:gpt-5.5",  # Optional override, defaults to main agent model
}
subagents = [research_subagent]

agent = create_deep_agent(
    model="openai:gpt-5.5",
    subagents=subagents,
)

from rich import print
result = agent.invoke({"messages": [{"role": "user", "content": "Hello"}]})
print(result)

# Backends

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import StateBackend

# By default, we provide a StateBackend
agent = create_deep_agent(model="openai:gpt-5.5")

# Under the hood, it looks like
agent2 = create_deep_agent(
    model="openai:gpt-5.5",
    backend=StateBackend(),
)

result = agent.invoke({"messages": [{"role": "user", "content": "Hello"}]})
from rich import print
print(result)

In [ ]:
import os

from deepagents import create_deep_agent
from langchain_runloop import RunloopSandbox
from runloop_api_client import RunloopSDK

from dotenv import load_dotenv
load_dotenv(override=True)

client = RunloopSDK(bearer_token=os.environ["RUNLOOP_API_KEY"])

devbox = client.devbox.create()
backend = RunloopSandbox(devbox=devbox)

agent = create_deep_agent(
    model="openai:gpt-5.5",
    system_prompt="You are a Python coding assistant with sandbox access.",
    backend=backend,
)

try:
    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "Create a small Python package and run pytest",
                }
            ]
        }
    )
finally:
    devbox.shutdown()

result = agent.invoke({"messages": [{"role": "user", "content": "Hello"}]})
from rich import print
print(result)

# Human-in-the-loop

In [ ]:
from langchain.tools import tool
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import MemorySaver


@tool
def remove_file(path: str) -> str:
    """Delete a file from the filesystem."""
    return f"Deleted {path}"


@tool
def fetch_file(path: str) -> str:
    """Read a file from the filesystem."""
    return f"Contents of {path}"


@tool
def notify_email(to: str, subject: str, body: str) -> str:
    """Send an email."""
    return f"Sent email to {to}"


# Checkpointer is REQUIRED for human-in-the-loop
checkpointer = MemorySaver()

agent = create_deep_agent(
    model="openai:gpt-5.5",
    tools=[remove_file, fetch_file, notify_email],
    interrupt_on={
        "remove_file": True,  # Default: approve, edit, reject, respond
        "fetch_file": False,  # No interrupts needed
        "notify_email": {"allowed_decisions": ["approve", "reject"]},  # No editing
    },
    checkpointer=checkpointer,  # Required!
)

session_config = {
    "configurable": {
        "thread_id": "chat_session_001"  # 自定义任意唯一字符串
    }
}

result = agent.invoke(
    {"messages": [{"role": "user", "content": "删除/Users/zhaojian/code/deepagents/examples/learn/test.txttest.txt文件"}]},
    config=session_config
)
from rich import print
print(result)

# SKILLS

In [5]:
from urllib.request import urlopen
from deepagents import create_deep_agent
from deepagents.backends import StateBackend
from deepagents.backends.utils import create_file_data
from langgraph.checkpoint.memory import MemorySaver

from dotenv import load_dotenv
load_dotenv(override=True)

checkpointer = MemorySaver()
backend = StateBackend()

skill_url = "https://raw.githubusercontent.com/langchain-ai/deepagents/refs/heads/main/libs/cli/examples/skills/langgraph-docs/SKILL.md"
with urlopen(skill_url) as response:
    skill_content = response.read().decode('utf-8')

skills_files = {
    "/skills/langgraph-docs/SKILL.md": create_file_data(skill_content),
}

agent = create_deep_agent(
    model="openai:gpt-5.5",
    backend=backend,
    skills=["/skills/"],
    checkpointer=checkpointer,
)

result = agent.invoke(
    {
        "messages": [{"role": "user", "content": "What is langgraph?"}],
        # Seed the default StateBackend's in-state filesystem (virtual paths must start with "/").
        "files": skills_files,
    },
    config={"configurable": {"thread_id": "12345"}},
)

from rich import print
print(result)

NameError: name 'load_dotenv' is not defined